# Oracle Manipulation and Flash Loans — The One-Transaction Heist

**Question:** how can a protocol execute every instruction correctly and still lose money? We will follow one deliberately tiny DeFi world in which a lending protocol treats an AMM's momentary price as truth. The arithmetic is real; the actors are imaginary, so nobody needs to call a lawyer.

**Scope:** this is a deterministic teaching model, not a recipe for attacking or operating a real protocol. It omits fees, liquidity providers, slippage limits, token transfers, external arbitrage, and real-world legal or economic consequences.


## Recap

Notebook 5 ended with a 10 ETH position backed by a $12,000 debt. At a sensible ETH price near $2,000, its collateral ratio is healthy. It also introduced the uncomfortable idea that a contract cannot discover market truth by itself: it must trust some input. Here the bad input is an on-chain AMM spot price that an attacker can temporarily move.

This notebook runs independently: it recreates the minimum premise instead of requiring a hidden notebook 5 kernel.


## 1. The whole attack, before the code

Inside **one atomic transaction**, the attacker will: (1) borrow ETH, (2) dump it into a small ETH/USD pool, (3) liquidate a victim whom the distorted price makes look unsafe, (4) use the received USD to buy the borrowed ETH back, (5) repay the principal, and (6) keep the seized collateral as surplus. This is a teaching model, not a recipe for a real protocol.

| Stage | What changes | Why it matters |
| --- | --- | --- |
| Borrow | attacker temporarily receives ETH | capital is available only for this transaction |
| Dump | AMM spot price falls | the lending rule sees a misleading input |
| Liquidate | attacker pays debt and receives collateral | the vulnerable rule follows its price source |
| Buy back and repay | borrowed principal returns to lender | any remaining ETH belongs to the attacker |


## 2. Atomic means everything commits or everything reverts

Atomicity means the transaction is all-or-nothing. If every required step succeeds, its state changes commit together. If one step raises an exception, our provider restores the AMM reserves, loans, and lender balance from a snapshot. Atomicity protects consistency; it does **not** decide whether the lending rule was economically sensible.

On Ethereum, a reverted transaction can still be included and consume gas. Gas is simply the meter for computation, and its fees are paid in ETH; that is all we need about it here.


## 3. The AMM: a market whose price lives in its reserves

A constant-product AMM keeps `x * y = k`. Here `x` is ETH reserve and `y` is USD reserve, so its displayed spot price is `USD reserve / ETH reserve`. The model omits trading fees, slippage limits, external arbitrage, and token decimals so that the reserve movement is visible. The next definition cell prints nothing; it creates the reserve-changing swaps whose output will reveal price impact.


In [ ]:
from dataclasses import dataclass


@dataclass
class AMMPool:
    """A constant-product (x*y=k) two-asset market maker.

    Attributes:
        eth_reserve: ETH held by the pool.
        usd_reserve: USD held by the pool.
    """

    eth_reserve: float
    usd_reserve: float

    def __post_init__(self) -> None:
        if self.eth_reserve <= 0 or self.usd_reserve <= 0:
            raise ValueError("AMM reserves must be positive.")

    @property
    def spot_price(self) -> float:
        """Current displayed price: USD reserve per unit of ETH reserve."""
        return self.usd_reserve / self.eth_reserve

    @property
    def constant_product(self) -> float:
        """The invariant `x * y` a swap should preserve (there are no fees here)."""
        return self.eth_reserve * self.usd_reserve

    def swap_eth_for_usd(self, eth_in: float) -> float:
        """Sell ETH into the pool, moving both reserves and the spot price.

        Args:
            eth_in: ETH sold into the pool. Must be positive.

        Returns:
            USD received in exchange.

        Raises:
            ValueError: If ``eth_in`` is not positive.
        """
        if eth_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_eth_reserve = self.eth_reserve + eth_in
        new_usd_reserve = k / new_eth_reserve
        usd_out = self.usd_reserve - new_usd_reserve
        self.eth_reserve, self.usd_reserve = new_eth_reserve, new_usd_reserve
        return usd_out

    def swap_usd_for_eth(self, usd_in: float) -> float:
        """Sell USD into the pool, moving both reserves and the spot price.

        Args:
            usd_in: USD sold into the pool. Must be positive.

        Returns:
            ETH received in exchange.

        Raises:
            ValueError: If ``usd_in`` is not positive.
        """
        if usd_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_usd_reserve = self.usd_reserve + usd_in
        new_eth_reserve = k / new_usd_reserve
        eth_out = self.eth_reserve - new_eth_reserve
        self.usd_reserve, self.eth_reserve = new_usd_reserve, new_eth_reserve
        return eth_out

## 4. Price impact: small pool, large footprint

We start with 50 ETH and $100,000: a $2,000/ETH spot price. We then sell 1 ETH and 40 ETH into **separate fresh pools**. Keeping the pools separate makes the comparison fair.

> Pause and predict: which sale moves the displayed price more, and does `x * y` remain essentially unchanged?


In [1]:
initial_pool = AMMPool(50.0, 100_000.0)
small_trade_pool = AMMPool(50.0, 100_000.0)
large_trade_pool = AMMPool(50.0, 100_000.0)

small_usd_out = small_trade_pool.swap_eth_for_usd(1.0)
large_usd_out = large_trade_pool.swap_eth_for_usd(40.0)
small_impact = (small_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
large_impact = (large_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
constant_product_preserved = abs(
    large_trade_pool.constant_product - initial_pool.constant_product
) < 1e-6

print(
    f"Initial pool: {initial_pool.eth_reserve:.2f} ETH and "
    f"${initial_pool.usd_reserve:,.2f}; spot price ${initial_pool.spot_price:,.2f}/ETH"
)
print(
    f"Small 1 ETH sale: receives ${small_usd_out:,.2f}; "
    f"price ${small_trade_pool.spot_price:,.2f}/ETH ({small_impact:.2f}%)"
)
print(
    f"Large 40 ETH sale: receives ${large_usd_out:,.2f}; "
    f"price ${large_trade_pool.spot_price:,.2f}/ETH ({large_impact:.2f}%)"
)
print(f"x * y preserved within floating-point tolerance: {constant_product_preserved}")


Initial pool: 50.00 ETH and $100,000.00; spot price $2,000.00/ETH
Small 1 ETH sale: receives $1,960.78; price $1,922.34/ETH (-3.88%)
Large 40 ETH sale: receives $44,444.44; price $617.28/ETH (-69.14%)
x * y preserved within floating-point tolerance: True


**Read the result:** the 40 ETH sale changes the AMM's displayed price by about 69%, far more than the 1 ETH sale. Nothing says ETH became 69% less valuable everywhere; this thin pool merely observed its own changed reserves. The constant product stays stable apart from ordinary floating-point rounding.


## 5. The vulnerable lending protocol

Our simplified lending protocol liquidates a loan whenever collateral value divided by debt is below 1.5. Its mistake is intentionally narrow: it reads `pool.spot_price` directly. Real protocols must also consider liquidation bonuses, partial liquidations, fees, token transfers, and much more; we omit those mechanics to isolate the bad price source. The following definition cell prints nothing; it gives the later examples one explicit decision rule to test.


In [ ]:
@dataclass(frozen=True)
class Loan:
    """A borrower's position: collateral posted against debt owed.

    Attributes:
        collateral_eth: ETH locked as collateral. Must be positive.
        debt_usd: Outstanding debt in USD. Must be positive.
    """

    collateral_eth: float
    debt_usd: float

    def __post_init__(self) -> None:
        if self.collateral_eth <= 0 or self.debt_usd <= 0:
            raise ValueError("Loan collateral and debt must be positive.")


class LoanNotFoundError(Exception):
    """Raised when a borrower has no open loan."""


class PositionNotLiquidatableError(Exception):
    """Raised when liquidation is attempted on a still-healthy position."""


class InsufficientRepaymentError(Exception):
    """Raised when a flash-loan action cannot return its borrowed principal."""


class LendingProtocol:
    """A toy lending protocol that liquidates undercollateralised loans.

    Its one deliberate flaw: by default it prices collateral from
    ``pool.spot_price`` -- a single, on-chain, manipulable number -- unless
    a ``price_source`` (e.g. a ``MedianOracle``) is supplied instead.
    """

    def __init__(
        self, pool: AMMPool, liquidation_ratio: float = 1.5, price_source=None
    ) -> None:
        """Configure the protocol's price source and liquidation threshold.

        Args:
            pool: The AMM this protocol reads a price from by default.
            liquidation_ratio: Minimum healthy collateral ratio. Must be
                positive.
            price_source: Optional object exposing a ``.price`` property.
                When given, it is trusted instead of ``pool.spot_price``.

        Raises:
            ValueError: If ``liquidation_ratio`` is not positive.
        """
        if liquidation_ratio <= 0:
            raise ValueError("Liquidation ratio must be positive.")
        self.pool = pool
        self.liquidation_ratio = liquidation_ratio
        self.price_source = price_source
        self.loans: dict[str, Loan] = {}

    def add_loan(self, borrower: str, loan: Loan) -> None:
        """Register an open loan for a borrower.

        Args:
            borrower: Non-empty borrower identifier.
            loan: The loan to register.

        Raises:
            ValueError: If ``borrower`` is empty.
        """
        if not borrower:
            raise ValueError("Borrower name must be non-empty.")
        self.loans[borrower] = loan

    def collateral_ratio(self, loan: Loan) -> float:
        """Return collateral value divided by debt, using the configured price source."""
        price = (
            self.price_source.price
            if self.price_source is not None
            else self.pool.spot_price
        )
        return loan.collateral_eth * price / loan.debt_usd

    def liquidate(self, borrower: str) -> Loan:
        """Seize and remove a borrower's loan if it is unhealthy.

        Args:
            borrower: The borrower to liquidate.

        Returns:
            The removed ``Loan``.

        Raises:
            LoanNotFoundError: If ``borrower`` has no open loan.
            PositionNotLiquidatableError: If the loan's collateral ratio
                is still at or above ``self.liquidation_ratio``.
        """
        if borrower not in self.loans:
            raise LoanNotFoundError(f"No loan for {borrower}.")
        loan = self.loans[borrower]
        if self.collateral_ratio(loan) >= self.liquidation_ratio:
            raise PositionNotLiquidatableError("Position is still healthy.")
        return self.loans.pop(borrower)

At the original AMM price, the victim has 10 ETH collateral and $12,000 debt.

> Pause and predict: is a 10 ETH position at $2,000/ETH above or below a 1.5 collateral-ratio threshold?


In [2]:
baseline_pool = AMMPool(50.0, 100_000.0)
baseline_protocol = LendingProtocol(baseline_pool)
victim_loan = Loan(collateral_eth=10.0, debt_usd=12_000.0)
baseline_protocol.add_loan("victim", victim_loan)
baseline_ratio = baseline_protocol.collateral_ratio(victim_loan)

print(f"At ${baseline_pool.spot_price:,.2f}/ETH, victim collateral ratio: {baseline_ratio:.2f}")
print("Liquidation threshold: 1.50; verdict: HEALTHY")


At $2,000.00/ETH, victim collateral ratio: 1.67
Liquidation threshold: 1.50; verdict: HEALTHY


**Read the result:** $20,000 of collateral divided by $12,000 debt is 1.67, so this position is healthy before the attack. The coming liquidation is not caused by the victim changing their loan; it is caused by the protocol trusting a temporary pool price.


## 6. Flash loans: huge capital, one-transaction expiry date

A flash-loan provider lends ETH only if the principal is available again by the end of the same transaction. This toy provider charges no fee so the accounting stays visible. It snapshots every modeled object it touches before the action and restores that snapshot if anything fails. The provider receives its principal exactly once; only the remaining ETH is attacker profit. The next definition cell prints nothing; it builds the snapshot and trace objects that will expose the committed and reverted paths.


In [ ]:
@dataclass(frozen=True)
class WorldSnapshot:
    """A pre-action copy of every modeled object a flash loan might mutate.

    Attributes:
        pool_eth: AMM ETH reserve before the action.
        pool_usd: AMM USD reserve before the action.
        loans: Copy of the lending protocol's open loans before the action.
        provider_eth: Flash-loan provider liquidity before the action.
    """

    pool_eth: float
    pool_usd: float
    loans: dict[str, Loan]
    provider_eth: float


@dataclass(frozen=True)
class AttackTrace:
    """A structured record of what a flash-loan attack actually did.

    Attributes:
        borrowed_eth: Principal borrowed for the transaction.
        usd_from_dump: USD received from selling the borrowed ETH.
        manipulated_price: AMM spot price immediately after the dump.
        victim_ratio: Victim's collateral ratio at the manipulated price.
        debt_paid_usd: USD paid to liquidate the victim.
        collateral_seized: ETH collateral seized from the victim.
        eth_bought_back: ETH bought back with the leftover USD.
        eth_before_repayment: Total attacker ETH before repaying principal.
        principal_repaid: ETH principal returned to the flash-loan provider.
    """

    borrowed_eth: float
    usd_from_dump: float
    manipulated_price: float
    victim_ratio: float
    debt_paid_usd: float
    collateral_seized: float
    eth_bought_back: float
    eth_before_repayment: float
    principal_repaid: float


class FlashLoanProvider:
    """Lends ETH that must be repaid before the same call returns.

    Snapshots every modeled object the action touches beforehand and
    restores that snapshot if the action raises or fails to repay --
    modelling atomic all-or-nothing execution without a real EVM.
    """

    def __init__(self, eth_available: float) -> None:
        """Set the provider's lendable liquidity.

        Args:
            eth_available: ETH the provider can lend. Must be positive.

        Raises:
            ValueError: If ``eth_available`` is not positive.
        """
        if eth_available <= 0:
            raise ValueError("Flash-loan liquidity must be positive.")
        self.eth_available = eth_available

    def execute(self, amount_eth, action, pool, protocol):
        """Lend ``amount_eth``, run ``action``, then commit or roll back.

        Args:
            amount_eth: ETH to lend. Must be positive and available.
            action: Callable taking the borrowed ETH amount and returning
                ``(eth_before_repayment, trace)``.
            pool: The AMM the action may mutate (snapshotted first).
            protocol: The lending protocol the action may mutate
                (snapshotted first).

        Returns:
            ``(profit_eth, trace)`` where ``profit_eth`` is whatever ETH
            remains after repaying the principal.

        Raises:
            ValueError: If ``amount_eth`` is not positive or exceeds
                available liquidity.
            InsufficientRepaymentError: If ``action`` cannot return at
                least ``amount_eth``. In this case (or any other
                exception from ``action``), ``pool``, ``protocol``, and
                this provider's balance are restored from the snapshot.
        """
        if amount_eth <= 0 or amount_eth > self.eth_available:
            raise ValueError("Flash-loan amount is unavailable.")
        snapshot = WorldSnapshot(
            pool.eth_reserve,
            pool.usd_reserve,
            dict(protocol.loans),
            self.eth_available,
        )
        self.eth_available -= amount_eth
        try:
            eth_before_repayment, trace = action(amount_eth)
            if eth_before_repayment < amount_eth:
                raise InsufficientRepaymentError(
                    f"Only {eth_before_repayment:.4f} ETH available to repay "
                    f"{amount_eth:.4f} ETH."
                )
            self.eth_available += amount_eth
            return eth_before_repayment - amount_eth, trace
        except Exception:
            pool.eth_reserve = snapshot.pool_eth
            pool.usd_reserve = snapshot.pool_usd
            protocol.loans = dict(snapshot.loans)
            self.eth_available = snapshot.provider_eth
            raise


def run_flash_attack(amount_eth: float, pool: AMMPool, protocol: LendingProtocol, victim: str) -> tuple[float, AttackTrace]:
    """Borrow, dump, liquidate, buy back, and report the resulting trace.

    Args:
        amount_eth: ETH to borrow and dump into ``pool``.
        pool: The AMM to manipulate.
        protocol: The lending protocol whose price source will be read.
        victim: Borrower name to liquidate once the price is manipulated.

    Returns:
        ``(eth_before_repayment, trace)`` -- total attacker ETH before
        repaying the flash-loan principal, and a structured trace of the
        attack's steps.

    Raises:
        PositionNotLiquidatableError: If the victim is not actually
            liquidatable at the manipulated price (propagates from
            ``protocol.liquidate``).
        InsufficientRepaymentError: If the liquidation proceeds leave no
            USD for the buyback swap.
    """
    print(f"STEP 1 — BORROW: {amount_eth:.2f} ETH arrives temporarily.")
    usd_from_dump = pool.swap_eth_for_usd(amount_eth)
    manipulated_price = pool.spot_price
    print(
        f"STEP 2 — DUMP: sell {amount_eth:.2f} ETH for ${usd_from_dump:,.2f}; "
        f"AMM price becomes ${manipulated_price:,.2f}/ETH."
    )
    victim_loan = protocol.loans[victim]
    victim_ratio = protocol.collateral_ratio(victim_loan)
    seized_loan = protocol.liquidate(victim)
    debt_paid_usd = seized_loan.debt_usd
    collateral_seized = seized_loan.collateral_eth
    print(
        f"STEP 3 — LIQUIDATE: victim ratio is {victim_ratio:.2f}; "
        f"pay ${debt_paid_usd:,.2f} debt and seize {collateral_seized:.2f} ETH."
    )
    usd_for_buyback = usd_from_dump - debt_paid_usd
    if usd_for_buyback <= 0:
        raise InsufficientRepaymentError("Liquidation leaves no USD for the buyback.")
    eth_bought_back = pool.swap_usd_for_eth(usd_for_buyback)
    eth_before_repayment = eth_bought_back + collateral_seized
    print(
        f"STEP 4 — BUY BACK: remaining ${usd_for_buyback:,.2f} buys back "
        f"{eth_bought_back:.2f} ETH; attacker holds {eth_before_repayment:.2f} ETH."
    )
    print(f"STEP 5 — REPAY: return {amount_eth:.2f} ETH principal to the provider.")
    trace = AttackTrace(
        borrowed_eth=amount_eth,
        usd_from_dump=usd_from_dump,
        manipulated_price=manipulated_price,
        victim_ratio=victim_ratio,
        debt_paid_usd=debt_paid_usd,
        collateral_seized=collateral_seized,
        eth_bought_back=eth_bought_back,
        eth_before_repayment=eth_before_repayment,
        principal_repaid=amount_eth,
    )
    return eth_before_repayment, trace

## 7. Run the heist

The flash-loan provider has 1,000 ETH, but the attacker borrows only 40 ETH. The pool starts at 50 ETH and $100,000, and the healthy victim loan is recreated from the recap.

> Pause and predict: after paying the victim's debt and using the remaining USD to reverse the dump, how much ETH remains once the 40 ETH principal is repaid?


In [3]:
attack_pool = AMMPool(50.0, 100_000.0)
attack_protocol = LendingProtocol(attack_pool)
attack_protocol.add_loan("victim", Loan(10.0, 12_000.0))
provider = FlashLoanProvider(1_000.0)

profit_eth, attack_trace = provider.execute(
    40.0,
    lambda borrowed_eth: run_flash_attack(
        borrowed_eth, attack_pool, attack_protocol, "victim"
    ),
    attack_pool,
    attack_protocol,
)
assert attack_trace.eth_before_repayment == (
    attack_trace.principal_repaid + profit_eth
)
assert provider.eth_available == 1_000.0

print("Attack transaction: COMMITTED")
print(f"Provider liquidity after repayment: {provider.eth_available:,.2f} ETH")
print(f"Attacker profit: {profit_eth:.2f} ETH")


STEP 1 — BORROW: 40.00 ETH arrives temporarily.
STEP 2 — DUMP: sell 40.00 ETH for $44,444.44; AMM price becomes $617.28/ETH.
STEP 3 — LIQUIDATE: victim ratio is 0.51; pay $12,000.00 debt and seize 10.00 ETH.
STEP 4 — BUY BACK: remaining $32,444.44 buys back 33.18 ETH; attacker holds 43.18 ETH.
STEP 5 — REPAY: return 40.00 ETH principal to the provider.
Attack transaction: COMMITTED
Provider liquidity after repayment: 1,000.00 ETH
Attacker profit: 3.18 ETH


**Read the result:** the dump produces $44,444.44 and makes the victim look unsafe at a 0.51 ratio. The liquidator must first pay the victim's $12,000 debt, leaving $32,444.44 for the reverse swap. That buys back about 33.18 ETH; with the wrongly seized 10 ETH collateral, the attacker has about 43.18 ETH before repayment. Repaying 40 ETH restores the provider to exactly 1,000 ETH, leaving about 3.18 ETH counted once as attacker profit. The transaction is atomically consistent and economically disastrous.


## 8. When one step fails, the whole transaction rewinds

Now change only the victim: Victim2 has 40 ETH collateral against the same $12,000 debt. Even after the 40 ETH dump, that position should remain healthy. The provider will still lend, the AMM will still be touched, and then liquidation will raise an exception.

> Pause and predict: when liquidation fails after the dump, which values should look exactly as they did before the flash-loan transaction?


In [4]:
failed_pool = AMMPool(50.0, 100_000.0)
failed_protocol = LendingProtocol(failed_pool)
failed_protocol.add_loan("Victim2", Loan(40.0, 12_000.0))
failed_provider = FlashLoanProvider(1_000.0)
failed_before = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
failed_observation: dict[str, float] = {}

def failed_attack_action(amount_eth: float):
    failed_pool.swap_eth_for_usd(amount_eth)
    failed_observation["price"] = failed_pool.spot_price
    failed_observation["ratio"] = failed_protocol.collateral_ratio(
        failed_protocol.loans["Victim2"]
    )
    failed_protocol.liquidate("Victim2")
    raise RuntimeError("Liquidation should have raised first.")

try:
    failed_provider.execute(
        40.0, failed_attack_action, failed_pool, failed_protocol
    )
except PositionNotLiquidatableError:
    print(
        f"Attempted dump moved AMM price to "
        f"${failed_observation['price']:,.2f}/ETH."
    )
    print(
        f"Victim2 ratio after dump: {failed_observation['ratio']:.2f}; "
        "liquidation is rejected."
    )
    print("Attack transaction: REVERTED")

failed_after = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
rollback_verified = failed_before == failed_after
print(f"Rollback complete: {rollback_verified}")


Attempted dump moved AMM price to $617.28/ETH.
Victim2 ratio after dump: 2.06; liquidation is rejected.
Attack transaction: REVERTED
Rollback complete: True


**Read the result:** the AMM price briefly reached $617.28, but Victim2 still had a 2.06 ratio, so the liquidation rule rejected the action. The exception made the provider restore the pool reserves, loan list, and lender balance from its snapshot.

## 9. Atomicity is a safety rail, not an honesty detector

Atomicity stops half-finished state from surviving. It cannot stop a fully completed transaction from exploiting a bad price rule, as the first attack did. On Ethereum, this reverted attempt could still be included and pay gas because computation was performed; our model keeps gas to that one sentence and focuses on state.


## 10. Defense: make the price harder to move than the profit is worth

A median of independent reports is not the same observation as a thin AMM's reserves. We will dump into a fresh AMM exactly as before, but configure the real lending protocol to ask a `MedianOracle` for its collateral-ratio and liquidation decisions. The next definition cell prints nothing; it creates the reports whose median will be compared with the manipulated pool price.


In [ ]:
import statistics


@dataclass(frozen=True)
class PriceReport:
    """One reported ETH/USD price from a named source.

    Attributes:
        source: Who or what reported this price.
        eth_price_usd: The reported price.
    """

    source: str
    eth_price_usd: float


class MedianOracle:
    """A price source that reports the median of several independent reports."""

    def __init__(self, reports: list[PriceReport]) -> None:
        """Store the reports this oracle will aggregate.

        Args:
            reports: Non-empty list of independent price reports.

        Raises:
            ValueError: If ``reports`` is empty.
        """
        if not reports:
            raise ValueError("At least one price report is required.")
        self.reports = reports

    @property
    def price(self) -> float:
        """Median reported price -- resists a single outlier report."""
        return statistics.median(report.eth_price_usd for report in self.reports)

The reports are $2,005, $1,995, and $2,000, so their median is $2,000.

> Pause and predict: after the same dump pushes the AMM to about $617/ETH, will the protocol liquidate a 10 ETH, $12,000 loan when its configured oracle remains near $2,000?


In [5]:
defense_oracle = MedianOracle(
    [
        PriceReport("exchange-A", 2_005.0),
        PriceReport("exchange-B", 1_995.0),
        PriceReport("reference-feed", 2_000.0),
    ]
)
defense_pool = AMMPool(50.0, 100_000.0)
defense_protocol = LendingProtocol(
    defense_pool, price_source=defense_oracle
)
defense_victim_loan = Loan(10.0, 12_000.0)
defense_protocol.add_loan("defense-victim", defense_victim_loan)
defense_pool.swap_eth_for_usd(40.0)
defense_ratio = defense_protocol.collateral_ratio(defense_victim_loan)

try:
    defense_protocol.liquidate("defense-victim")
except PositionNotLiquidatableError:
    defense_rejected = True
else:
    defense_rejected = False

def defense_verdict(defense_rejected: bool) -> str:
    return "liquidation rejected" if defense_rejected else "liquidation executed"


print(f"Manipulated AMM spot price: ${defense_pool.spot_price:,.2f}/ETH")
print(
    f"Median oracle price: ${defense_oracle.price:,.2f}/ETH; "
    f"protocol ratio: {defense_ratio:.2f}"
)
print(f"Defense result: {defense_verdict(defense_rejected)}")


Manipulated AMM spot price: $617.28/ETH
Median oracle price: $2,000.00/ETH; protocol ratio: 1.67
Defense result: liquidation rejected


**Read the result:** the AMM really was moved to $617.28, but `LendingProtocol.collateral_ratio()` read the median oracle's $2,000 price and `LendingProtocol.liquidate()` therefore rejected the healthy loan. Median feeds, time-weighted average prices, deeper liquidity, freshness limits, circuit breakers, and conservative parameters are layers of defense, not silver bullets.


## 11. Back onto the blockchain: reverted transactions still leave receipts

Earlier notebooks showed blocks forming an accepted history. A receipt adds one more useful distinction: inclusion says a transaction was recorded and executed; its status says whether its state changes survived. The next definition cell prints nothing; it imports `Validator`, `Block`, and `Blockchain` from `blockchain_lib.pos` (as notebook 5 also does) so this notebook plugs into the same chain abstraction instead of faking one, plus a `TransactionReceipt` that references a real chain block. The following demonstration will print three included receipts and their distinct statuses.


In [ ]:
import sys
from pathlib import Path


def _repo_root(marker: str = "pyproject.toml") -> Path:
    """Walk upward from the current working directory to find the repo root."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not find {marker} above {Path.cwd()}")


_ROOT = _repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from blockchain_lib.pos import Block, Blockchain, Validator


@dataclass(frozen=True)
class TransactionReceipt:
    """Per-transaction execution result for a block already on the chain.

    A block's presence in ``chain.chain`` means the transaction was
    *included*; ``status`` says separately whether its state changes
    *survived* -- the same distinction real chains keep between a
    transaction being mined and it succeeding.

    Attributes:
        block: The chain block this transaction was included in.
        transaction: Human-readable description, for demo printing.
        status: ``"SUCCESS"`` or ``"REVERTED"``.
        gas_used: Gas consumed by execution, regardless of status.
        state_effect: What (if anything) changed in modeled state.
    """

    block: Block
    transaction: str
    status: str
    gas_used: int
    state_effect: str

> Pause and predict: which timeline row was included in a block but left no modeled state change?


In [6]:
validators = [Validator("sequencer", stake=1)]
chain = Blockchain(validators)

block_1, _ = chain.add_block("Victim opens loan")
block_2, _ = chain.add_block("Oracle-manipulation attack")
block_3, _ = chain.add_block("Attack against Victim2")

receipts = [
    TransactionReceipt(block_1, "Victim opens loan", "SUCCESS", 52_000, "Loan created"),
    TransactionReceipt(
        block_2, "Oracle-manipulation attack", "SUCCESS", 310_000,
        "Attack state committed"
    ),
    TransactionReceipt(
        block_3, "Attack against Victim2", "REVERTED", 185_000, "No state change"
    ),
]

print("Block | Transaction                 | Status   | Gas used | State effect")
for receipt in receipts:
    print(
        f"{receipt.block.index:>5} | {receipt.transaction:<27} "
        f"| {receipt.status:<8} | {receipt.gas_used:>8,} | {receipt.state_effect}"
    )
print("Included does not mean succeeded")

chain_valid, chain_message = chain.is_valid()
print(f"Chain valid? {chain_valid} -- {chain_message}")

Block | Transaction                 | Status   | Gas used | State effect
    1 | Victim opens loan           | SUCCESS  |   52,000 | Loan created
    2 | Oracle-manipulation attack  | SUCCESS  |  310,000 | Attack state committed
    3 | Attack against Victim2      | REVERTED |  185,000 | No state change
Included does not mean succeeded
Chain valid? True -- Chain is valid.


**Read the result:** the first two transactions were included and their state changes committed. The third was also included and did work, but its `REVERTED` status means the state snapshot survived instead. Consensus records both receipts; execution status determines whether the attempted changes remain in canonical state.

## Takeaways

- An AMM spot price reflects its own reserves, not guaranteed market truth.
- A flash loan makes temporary capital available, but demands its principal back before the transaction ends.
- Atomicity prevents half-finished state, not a completed exploit of a bad rule.
- Independent, fresh price sources make one temporary AMM movement less persuasive.
- Consensus can faithfully record successful and reverted execution receipts; it cannot make a weak oracle truthful.
